In [4]:
import numpy as np

class RNN:
    def __init__(self, input_size, hidden_size, output_size):
        # Weights
        self.Wxh = np.random.randn(hidden_size, input_size) * 0.01
        self.Whh = np.random.randn(hidden_size, hidden_size) * 0.01
        self.Why = np.random.randn(output_size, hidden_size) * 0.01

        # Biases
        self.bh = np.zeros((hidden_size, 1))
        self.by = np.zeros((output_size, 1))

    def forward(self, inputs):
        h, y = {}, {}
        h[-1] = np.zeros((self.Wxh.shape[0], 1))

        # Forward pass
        for t in range(len(inputs)):
            x = np.array([[inputs[t]]])  # Convert to 2D array
            h[t] = np.tanh(np.dot(self.Wxh, x) + np.dot(self.Whh, h[t - 1]) + self.bh)
            y[t] = np.dot(self.Why, h[t]) + self.by

        self.last_inputs = inputs
        self.last_hs = h
        self.last_ys = y

        return y

    def backward(self, dY):
        n = len(self.last_inputs)

        # Initialize gradients
        dWxh = np.zeros_like(self.Wxh)
        dWhh = np.zeros_like(self.Whh)
        dWhy = np.zeros_like(self.Why)
        dbh = np.zeros_like(self.bh)
        dby = np.zeros_like(self.by)
        dhnext = np.zeros_like(self.last_hs[0])

        for t in reversed(range(n)):
            dy = np.copy(dY[t])
            dWhy += np.dot(dy, self.last_hs[t].T)
            dby += dy
            dh = np.dot(self.Why.T, dy) + dhnext
            dhraw = (1 - self.last_hs[t] * self.last_hs[t]) * dh
            dbh += dhraw
            dWxh += np.dot(dhraw, np.array([[self.last_inputs[t]]]).T)
            dWhh += np.dot(dhraw, self.last_hs[t - 1].T)
            dhnext = np.dot(self.Whh.T, dhraw)

        # Clip to prevent exploding gradients
        for dparam in [dWxh, dWhh, dWhy, dbh, dby]:
            np.clip(dparam, -5, 5, out=dparam)

        return dWxh, dWhh, dWhy, dbh, dby


In [6]:
def create_data(n, seq_length):
    inputs = np.random.randn(n, seq_length)
    targets = np.sum(inputs, axis=1)
    return inputs, targets


In [13]:
def mse_loss(y_true, y_pred):
    return ((y_true - y_pred) ** 2).mean()

def derivative_mse_loss(y_true, y_pred):
    return 2 * (y_pred - y_true) / y_true.size


# Hyperparameters
seq_length = 5
hidden_size = 50
output_size = 1
learning_rate = 0.001
epochs = 100

# Initialize our RNN
input_size = 1  # Since we're looking at one number at a time in the sequence
rnn = RNN(input_size, hidden_size, output_size)

# Create some data
n = 1000  # Number of samples
X, Y = create_data(n, seq_length)

# Training loop
for epoch in range(epochs):
    epoch_loss = 0
    for i in range(n):
        inputs = X[i]
        target = np.array([[Y[i]]])  # Target is a 2D array with a single value

        # Forward pass
        ys = rnn.forward(inputs)
        loss = mse_loss(target, ys[-1])
        epoch_loss += loss

        # Backward pass
        dY = {t: derivative_mse_loss(target, ys[t]) for t in range(seq_length)}
        dWxh, dWhh, dWhy, dbh, dby = rnn.backward(dY)

        # Update weights and biases
        for param, dparam in zip([rnn.Wxh, rnn.Whh, rnn.Why, rnn.bh, rnn.by],
                                 [dWxh, dWhh, dWhy, dbh, dby]):
            param -= learning_rate * dparam

    if epoch % 10 == 0:
        print(f'Epoch {epoch}, Loss: {epoch_loss / n}')


KeyError: -1

In [9]:
# Create test data
test_n = 100
test_X, test_Y = create_data(test_n, seq_length)

# Test loop
test_loss = 0
for i in range(test_n):
    inputs = test_X[i]
    target = np.array([[test_Y[i]]])

    # Forward pass
    ys = rnn.forward(inputs)
    loss = mse_loss(target, ys[-1])
    test_loss += loss

print(f'Test Loss: {test_loss / test_n}')


KeyError: -1